In [4]:
import os
from dotenv import load_dotenv

load_dotenv()
DATABASE_URL = os.getenv("DATABASE_URL")

DATABASE_URL

ModuleNotFoundError: No module named 'dotenv'

In [3]:
import psycopg2 as pg2
import pandas as pd
from datetime import datetime

def filterDateConstraints(df):
    curDate = pd.Timestamp(datetime.now().date())
    one_month = pd.Timedelta(days=31)
    one_year = pd.Timedelta(days=365)

    df['start_date'] = pd.to_datetime(df['start_date'])
    df['end_date'] = pd.to_datetime(df['end_date'])

    filtered = df[
        (df['start_date'] > curDate) &
        (df['end_date'] >= df['start_date'] + one_month) &
        (df['end_date'] <= df['start_date'] + one_year)
    ]
    return filtered

def filteTenantAgeConstraint(df):
    filtered = df[
         (df['age'] >= 18)
    ]
    return filtered

def bulkInsert(sheetName, dataFilepath, reset=False):
    df = pd.read_excel(dataFilepath, sheet_name=sheetName, engine='openpyxl')
    # if sheetName.lower() == 'listings' or sheetName.lower() == 'renter_profiles':
    #     df = filterDateConstraints(df)
    # if sheetName.lower() == 'renter_profiles':
    #     df = filteTenantAgeConstraint(df)

    cols = list(df.columns)
    query = f"""
        INSERT INTO {sheetName} ({", ".join(cols)})
        VALUES ({", ".join(["%s"] * len(cols))})
        ON CONFLICT DO NOTHING;
    """
    
    insertValues = list(df.itertuples(index=False, name=None))
    print(query)

    with pg2.connect(DATABASE_URL) as conn:
        with conn.cursor() as cur:
            if reset:
                cur.execute(f"TRUNCATE TABLE {sheetName} RESTART IDENTITY CASCADE")
            for row in insertValues:
                try:
                    cur.execute(query, row)
                except Exception as e:
                    print(f"error: {e}\nrow: {row}")
        conn.commit()


ModuleNotFoundError: No module named 'psycopg2'

In [2]:
FILEPATH = "./data/sample.xlsx"

excel = pd.ExcelFile(FILEPATH)
sheetNames = excel.sheet_names

for sheet in sheetNames:
    bulkInsert(sheet, FILEPATH, True)

NameError: name 'pd' is not defined